In [ ]:
from inverse_kinematics import IK
import numpy as np
from scipy.stats.qmc import Halton
from itertools import product
from typing import List, Tuple

# ========== 初始化真实 IK 求解器 ==========
ik_solver = IK('arm')
ik_solver.setLinkLength(L1=ik_solver.l1 + 0.75, L4=ik_solver.l4 - 0.15)

# ========== Step 1: Halton 采样函数 ==========
def halton_sample(n, x_range, y_range, z_range):
    sampler = Halton(d=3, scramble=True)
    sample = sampler.random(n=n)
    x = x_range[0] + (x_range[1] - x_range[0]) * sample[:, 0]
    y = y_range[0] + (y_range[1] - y_range[0]) * sample[:, 1]
    z = z_range[0] + (z_range[1] - z_range[0]) * sample[:, 2]
    return list(zip(x, y, z))

# ========== Step 2: 用真实 IK 验证 ==========
def ik_valid(pos: Tuple[float, float, float], alpha=0) -> bool:
    result = ik_solver.getRotationAngle(pos, alpha)
    return result is not False

# ========== Step 3: 构造组合并验证 ==========
def generate_valid_pairs_from_halton(
    n_pick=100, n_place=100,
    pick_box=((-30, -10), (-10, 10), (5, 10)),
    place_box=((10, 30), (-10, 10), (5, 10)),
    alpha=0
) -> List[Tuple[Tuple[float, float, float], Tuple[float, float, float]]]:
    print(f"🎯 Step 1: 采样 pick 和 place 点各 {n_pick} 个")
    pick_points = halton_sample(n_pick, *pick_box)
    place_points = halton_sample(n_place, *place_box)

    print(f"🔗 Step 2: 构造组合 ×{len(pick_points)} × {len(place_points)} = {len(pick_points) * len(place_points)} 个候选任务对")
    all_pairs = list(product(pick_points, place_points))

    print("✅ Step 3: 用 IK 验证合法性...")
    valid_pairs = []
    for pick, place in all_pairs:
        if ik_valid(pick, alpha) and ik_valid(place, alpha):
            valid_pairs.append((pick, place))

    print(f"✅ 获得合法任务对: {len(valid_pairs)} / {len(all_pairs)}")
    return valid_pairs


In [ ]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import json
import os

def select_diverse_samples_by_6d(
    valid_pairs,
    n_output=1000,
    mode='closest',
    return_distance=True,
    save_csv_path=None,
    save_json_path=None,
    plot=True,
    pca_title='6D Task Pair PCA Distribution'
):
    """
    从合法任务对中选出有代表性的样本（聚类于6D空间），支持可视化、导出

    Parameters:
        valid_pairs: [(pick_xyz, place_xyz), ...]
        n_output: 输出任务数量
        mode: 'first', 'random', 'closest'（从簇中取样方式）
        return_distance: 是否计算每个任务对的路径长度
        save_csv_path: 如设定则保存 CSV
        save_json_path: 如设定则保存 JSON
        plot: 是否绘制PCA图
        pca_title: 图标题
    Returns:
        selected_pairs: [(pick, place, distance)] or [(pick, place)]
    """
    if len(valid_pairs) < n_output:
        print(f"⚠️ 样本数量不足，返回全部合法点对。")
        n_output = len(valid_pairs)

    X = np.array([list(pick) + list(place) for pick, place in valid_pairs])
    kmeans = MiniBatchKMeans(n_clusters=n_output, random_state=0)
    kmeans.fit(X)

    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    selected = []
    for i in range(n_output):
        indices = np.where(labels == i)[0]
        if len(indices) == 0:
            continue

        if mode == 'first':
            idx = indices[0]
        elif mode == 'random':
            idx = np.random.choice(indices)
        elif mode == 'closest':
            dists = np.linalg.norm(X[indices] - centers[i], axis=1)
            idx = indices[np.argmin(dists)]
        else:
            raise ValueError("mode must be 'first', 'random', or 'closest'")

        pick, place = valid_pairs[idx]
        if return_distance:
            dist = np.linalg.norm(np.array(pick) - np.array(place))
            selected.append((pick, place, dist))
        else:
            selected.append((pick, place))

    # 可视化 PCA 分布
    if plot:
        print("📊 Generating PCA visualization...")
        pca = PCA(n_components=2)
        X_2d = pca.fit_transform(X)
        plt.figure(figsize=(8, 6))
        plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, s=5, cmap='tab20', alpha=0.3)
        plt.title(pca_title)
        plt.xlabel('PCA1')
        plt.ylabel('PCA2')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # 保存 CSV
    if save_csv_path:
        print(f"💾 Saving to CSV: {save_csv_path}")
        data = [
            list(pick) + list(place) + ([dist] if return_distance else [])
            for (pick, place, dist) in selected
        ] if return_distance else [
            list(pick) + list(place)
            for (pick, place) in selected
        ]
        cols = ['pick_x', 'pick_y', 'pick_z', 'place_x', 'place_y', 'place_z']
        if return_distance:
            cols.append('distance')
        df = pd.DataFrame(data, columns=cols)
        os.makedirs(os.path.dirname(save_csv_path), exist_ok=True)
        df.to_csv(save_csv_path, index=False)

    # 保存 JSON
    if save_json_path:
        print(f"💾 Saving to JSON: {save_json_path}")
        out_data = []
        for item in selected:
            if return_distance:
                pick, place, dist = item
                out_data.append({'pick': pick, 'place': place, 'distance': dist})
            else:
                pick, place = item
                out_data.append({'pick': pick, 'place': place})
        os.makedirs(os.path.dirname(save_json_path), exist_ok=True)
        with open(save_json_path, 'w') as f:
            json.dump(out_data, f, indent=2)

    return selected


In [ ]:
from pathlib import Path

valid_pairs = generate_valid_pairs_from_halton(n_pick=100, n_place=100)

# 然后用 step 4 的聚类挑选代表样本（前面写好的函数）
samples = select_diverse_samples_by_6d(
    valid_pairs,
    n_output=1000,
    mode='closest',
    return_distance=True,
    save_csv_path='output/representative_tasks.csv',
    save_json_path='output/representative_tasks.json',
    plot=True
)


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats.qmc import Halton
import numpy as np

# Halton 采样函数
def halton_sample(n, x_range, y_range, z_range):
    sampler = Halton(d=3, scramble=True)
    sample = sampler.random(n=n)
    x = x_range[0] + (x_range[1] - x_range[0]) * sample[:, 0]
    y = y_range[0] + (y_range[1] - y_range[0]) * sample[:, 1]
    z = z_range[0] + (z_range[1] - z_range[0]) * sample[:, 2]
    return x, y, z

# 设置采样范围（Pick 区域示例）
x_range = (-30, -10)
y_range = (-10, 10)
z_range = (5, 10)

# 执行 Halton 采样
x, y, z = halton_sample(200, x_range, y_range, z_range)

# 3D 可视化
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z, c='green', s=30, alpha=0.8)
ax.set_xlabel('X (cm)')
ax.set_ylabel('Y (cm)')
ax.set_zlabel('Z (cm)')
ax.set_title('Halton Sampling in Pick Region')
ax.view_init(elev=25, azim=135)

plt.tight_layout()

plt.show()


In [ ]:
# 设置你的案例的实际空间范围（Pick 区域）
x_range = (-30, -10)
y_range = (-10, 10)
z_range = (5, 10)

# 将 Halton 点映射到实际坐标范围
def map_to_range(x, x_range):
    return x_range[0] + (x_range[1] - x_range[0]) * x

x_real = map_to_range(x_vals, x_range)
y_real = map_to_range(y_vals, y_range)
z_real = map_to_range(z_vals, z_range)

# 创建新的 3D 图形
fig_real = plt.figure(figsize=(8, 6))
ax_real = fig_real.add_subplot(111, projection='3d')
ax_real.set_xlim(x_range)
ax_real.set_ylim(y_range)
ax_real.set_zlim(z_range)
ax_real.set_xlabel("X (cm)")
ax_real.set_ylabel("Y (cm)")
ax_real.set_zlabel("Z (cm)")
ax_real.set_title("Halton 3D Sampling in Real Pick Region")

scat_real = ax_real.scatter([], [], [], c='green', s=30)

# 初始化函数
def init_real():
    scat_real._offsets3d = ([], [], [])
    return scat_real,

# 更新函数
def update_real(frame):
    scat_real._offsets3d = (
        x_real[:frame],
        y_real[:frame],
        z_real[:frame]
    )
    return scat_real,

# 创建动画
ani_real = animation.FuncAnimation(
    fig_real, update_real, frames=np.arange(1, n_points+1),
    init_func=init_real, blit=False, repeat=False, interval=100
)

# 显示动画
plt.close(fig_real)

In [ ]:
from IPython.display import HTML
HTML(ani_real.to_jshtml())


In [ ]:
with open("halton_animation_real.html", "w") as f:
    f.write(ani_real.to_jshtml())


In [ ]:
ani_real.save("halton_sampling_real.gif", writer="pillow", fps=10)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.animation import PillowWriter

# -------------------------------
# ✅ 生成 Halton 序列（Base = 2）
# -------------------------------
def halton_base_2(n):
    result = []
    for i in range(1, n + 1):
        f = 0.5
        r = 0
        index = i
        while index > 0:
            r += f * (index % 2)
            index //= 2
            f /= 2
        result.append(r)
    return result

# 参数
n_points = 20
halton_vals = halton_base_2(n_points)

# --------------------------------------
# ✅ 可视化构造过程（动画 + 静态 + 保存）
# --------------------------------------

# 创建图形
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_title("Halton Sequence Construction (Base = 2)", fontsize=14)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.get_yaxis().set_visible(False)
ax.set_xlabel("Value", fontsize=10)

# 灰色参考线
ax.hlines(0.5, 0, 1, color='gray', linestyle='--', alpha=0.3)

# 初始化点图
scat = ax.scatter([], [], c='tab:blue', s=50)
texts = []

# 初始化函数
def init():
    scat.set_offsets(np.empty((0, 2)))
    return scat,

# 更新函数（每一帧添加一个点）
def update(frame):
    x = halton_vals[:frame]
    y = [0.5] * frame
    scat.set_offsets(np.c_[x, y])

    # 动态更新数字标签
    for t in texts:
        t.remove()
    texts.clear()
    for i, val in enumerate(x):
        t = ax.text(val, 0.55, f"{i+1}", ha='center', fontsize=10)
        texts.append(t)

    return scat, *texts

# 创建动画
ani = animation.FuncAnimation(
    fig, update, frames=np.arange(1, n_points + 1),
    init_func=init, blit=False, repeat=False, interval=500
)

# ✅ 保存 HTML 交互动画
with open("halton_1d_animation.html", "w") as f:
    f.write(ani.to_jshtml())

# ✅ 可选：保存为 gif 动画（用于 PPT）
ani.save("halton_1d_animation.gif", writer=PillowWriter(fps=2))

# 显示动画（用于交互）
plt.close(fig)
from IPython.display import HTML
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats.qmc import Halton
import matplotlib.animation as animation

# 设置空间范围
x_range = (-30, -10)
y_range = (-10, 10)
z_range = (5, 10)

# 对比步长与数量一致
step = 4
x_vals = np.arange(x_range[0], x_range[1] + 1e-6, step)
y_vals = np.arange(y_range[0], y_range[1] + 1e-6, step)
z_vals = np.arange(z_range[0], z_range[1] + 1e-6, step)
grid_points = np.array(np.meshgrid(x_vals, y_vals, z_vals)).T.reshape(-1, 3)

n_halton = len(grid_points)
sampler = Halton(d=3, scramble=True)
halton_raw = sampler.random(n=n_halton)

# 映射 Halton 点到真实范围
def map_to_range(x, r): return r[0] + (r[1] - r[0]) * x

halton_points = np.stack([
    map_to_range(halton_raw[:, 0], x_range),
    map_to_range(halton_raw[:, 1], y_range),
    map_to_range(halton_raw[:, 2], z_range)
], axis=-1)

# 创建动画图形
fig = plt.figure(figsize=(12, 6))

# 左：Grid
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title("Grid Sampling")
ax1.set_xlim(x_range)
ax1.set_ylim(y_range)
ax1.set_zlim(z_range)
ax1.set_xlabel("X")
ax1.set_ylabel("Y")
ax1.set_zlabel("Z")
scat1 = ax1.scatter([], [], [], c='blue', s=15)

# 右：Halton
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title("Halton Sampling")
ax2.set_xlim(x_range)
ax2.set_ylim(y_range)
ax2.set_zlim(z_range)
ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_zlabel("Z")
scat2 = ax2.scatter([], [], [], c='green', s=15)

# 初始化函数
def init():
    scat1._offsets3d = ([], [], [])
    scat2._offsets3d = ([], [], [])
    return scat1, scat2

# 更新函数
def update(frame):
    g = grid_points[:frame]
    h = halton_points[:frame]
    scat1._offsets3d = (g[:, 0], g[:, 1], g[:, 2])
    scat2._offsets3d = (h[:, 0], h[:, 1], h[:, 2])
    return scat1, scat2

ani = animation.FuncAnimation(
    fig, update, frames=np.arange(1, n_halton + 1, 30),
    init_func=init, blit=False, repeat=False, interval=100
)

# 保存动画
from matplotlib.animation import PillowWriter
ani.save("/mnt/data/grid_vs_halton.gif", writer=PillowWriter(fps=10))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

# 设置空间范围（Pick 区域）
x_range = (-30, -10)
y_range = (-10, 10)
z_range = (5, 10)

# 定义不同粒度的网格步长（表示维度爆炸）
steps = [15, 10, 6, 4, 2]  # 越小越密，点数爆炸

# 初始化动画数据
frames = []
labels = []

for step in steps:
    x_vals = np.arange(x_range[0], x_range[1] + 1e-6, step)
    y_vals = np.arange(y_range[0], y_range[1] + 1e-6, step)
    z_vals = np.arange(z_range[0], z_range[1] + 1e-6, step)

    grid = np.array(np.meshgrid(x_vals, y_vals, z_vals)).T.reshape(-1, 3)
    frames.append(grid)
    labels.append(f"{step}cm step → {len(grid)} pts")

# 创建动画图像
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

scat = ax.scatter([], [], [], s=10, c='purple')
title = ax.set_title("")

ax.set_xlim(x_range)
ax.set_ylim(y_range)
ax.set_zlim(z_range)
ax.set_xlabel("X (cm)")
ax.set_ylabel("Y (cm)")
ax.set_zlabel("Z (cm)")

def init():
    scat._offsets3d = ([], [], [])
    title.set_text("")
    return scat, title

def update(frame_idx):
    data = frames[frame_idx]
    scat._offsets3d = (data[:, 0], data[:, 1], data[:, 2])
    title.set_text(f"Grid Step = {steps[frame_idx]} cm | {len(data)} points")
    return scat, title

ani = animation.FuncAnimation(
    fig, update, frames=len(frames),
    init_func=init, blit=False, repeat=True, interval=1500
)

# 保存为 GIF
from matplotlib.animation import PillowWriter
ani.save("/mnt/data/grid_sampling_explosion.gif", writer=PillowWriter(fps=1))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

# Assuming a CSV file is uploaded separately, simulate reading it
csv_path = r"F:\OneDrive - CentraleSupelec\PhD\Paper-Fault_detection\codes\data\representative_tasks_300_43.csv"  # Expected CSV format with headers: pick_x, pick_y, pick_z

# Read CSV file
df = pd.read_csv(csv_path)

# Create a 3D animation
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

xdata, ydata, zdata = [], [], []
sc = ax.scatter([], [], [], c='blue')

def init():
    ax.set_xlim(df['pick_x'].min(), df['pick_x'].max())
    ax.set_ylim(df['pick_y'].min(), df['pick_y'].max())
    ax.set_zlim(df['pick_z'].min(), df['pick_z'].max())
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    return sc,

def update(frame):
    xdata.append(df.loc[frame, 'pick_x'])
    ydata.append(df.loc[frame, 'pick_y'])
    zdata.append(df.loc[frame, 'pick_z'])
    ax.clear()
    ax.set_xlim(df['pick_x'].min(), df['pick_x'].max())
    ax.set_ylim(df['pick_y'].min(), df['pick_y'].max())
    ax.set_zlim(df['pick_z'].min(), df['pick_z'].max())
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.scatter(xdata, ydata, zdata, c='blue')
    return sc,

ani = animation.FuncAnimation(fig, update, frames=len(df), init_func=init, blit=False, interval=100)

# Save the animation as a gif
gif_path = "points_animation.gif"
ani.save(gif_path, writer='pillow')

gif_path


# 从这里开始

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats.qmc import Halton
from itertools import product
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ✅ 导入真实 IK 模块
from inverse_kinematics import IK

# ✅ 设置随机种子（可选）
seed = 45
np.random.seed(seed)

# ✅ 初始化 IK
ik_solver = IK('arm')
ik_solver.setLinkLength(L1=ik_solver.l1 + 0.75, L4=ik_solver.l4 - 0.15)

def is_valid(pos, alpha=0):
    result = ik_solver.getRotationAngle(pos, alpha)
    if result is False:
        print(f"❌ 无解 IK 点: {pos}")
    return result is not False

# ✅ Halton 采样函数
def halton_sample(n, x_range, y_range, z_range, seed=seed):
    sampler = Halton(d=3, scramble=True, seed=seed)  # ✅ 固定种子
    sample = sampler.random(n=n)

    x = x_range[0] + (x_range[1] - x_range[0]) * sample[:, 0]
    y = y_range[0] + (y_range[1] - y_range[0]) * sample[:, 1]
    z = z_range[0] + (z_range[1] - z_range[0]) * sample[:, 2]

    # 四舍五入变整数
    x = np.round(x, decimals=2)
    y = np.round(y, decimals=2) 
    z = np.round(z, decimals=2)

    return list(zip(x, y, z))

# ✅ Random 采样函数：真正空间内随机生成点
def random_sample(n, x_range, y_range, z_range, seed=seed):
    rng = np.random.default_rng(seed)

    x = rng.uniform(x_range[0], x_range[1], size=n)
    y = rng.uniform(y_range[0], y_range[1], size=n)
    z = rng.uniform(z_range[0], z_range[1], size=n)

    x = np.round(x, decimals=2)
    y = np.round(y, decimals=2)
    z = np.round(z, decimals=2)

    return list(zip(x, y, z))

# ✅ 设置 Pick / Place 区域
PickR_Px = [-25, -10]
PickR_Py = [10, 25]
PickR_Pz = [6, 18]

PlaceR_Px = [10, 25]
PlaceR_Py = [10, 25]
PlaceR_Pz = [6, 18]

# ✅ 新增：保存 Pick / Place 100 点
import pandas as pd
from pathlib import Path

save_dir = Path("saved_sampling_points")
save_dir.mkdir(parents=True, exist_ok=True)

def save_pick_place_points(method_name, pick_points, place_points, seed=seed, n_points=100):
    pick_df = pd.DataFrame(pick_points, columns=["x", "y", "z"])
    pick_df.insert(0, "point_id", np.arange(1, len(pick_df) + 1))

    place_df = pd.DataFrame(place_points, columns=["x", "y", "z"])
    place_df.insert(0, "point_id", np.arange(1, len(place_df) + 1))

    pick_csv_path = save_dir / f"{method_name}_pick_points_seed{seed}_{n_points}.csv"
    place_csv_path = save_dir / f"{method_name}_place_points_seed{seed}_{n_points}.csv"

    pick_df.to_csv(pick_csv_path, index=False)
    place_df.to_csv(place_csv_path, index=False)

    print(f"✅ {method_name} Pick 100 points saved to: {pick_csv_path}")
    print(f"✅ {method_name} Place 100 points saved to: {place_csv_path}")

# ✅ 新增：生成 Halton 点
def generate_halton_points(n_points=100, seed=seed):
    pick_points = halton_sample(n_points, PickR_Px, PickR_Py, PickR_Pz, seed=seed)
    print(f"✅ 采样 Pick 点：{pick_points}")

    place_points = halton_sample(n_points, PlaceR_Px, PlaceR_Py, PlaceR_Pz, seed=seed)
    print(f"✅ 采样 Place 点：{place_points}")

    save_pick_place_points("halton", pick_points, place_points, seed=seed, n_points=n_points)

    return pick_points, place_points

# ✅ 新增：生成 Random 点
def generate_random_points(n_points=100, seed=seed):
    pick_points = random_sample(n_points, PickR_Px, PickR_Py, PickR_Pz, seed=seed)
    print(f"✅ 随机采样 Pick 点：{pick_points}")

    place_points = random_sample(n_points, PlaceR_Px, PlaceR_Py, PlaceR_Pz, seed=seed + 1)
    print(f"✅ 随机采样 Place 点：{place_points}")

    save_pick_place_points("random", pick_points, place_points, seed=seed, n_points=n_points)

    return pick_points, place_points


# ============================================================
# ✅ 这里选择调用哪个采样方法
# ============================================================

n_points = 100

# 可选："halton" 或 "random"
sampling_method = "random"

if sampling_method == "halton":
    pick_points, place_points = generate_halton_points(n_points=n_points, seed=seed)
elif sampling_method == "random":
    pick_points, place_points = generate_random_points(n_points=n_points, seed=seed)
else:
    raise ValueError("sampling_method 只能是 'halton' 或 'random'")


# ✅ 构造组合并 IK 验证
valid_pairs = []
for pick, place in product(pick_points, place_points):
    if is_valid(pick) and is_valid(place):
        valid_pairs.append((pick, place))


import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'Arial',  # 或 'Times New Roman'
    'axes.titlesize': 14,
    'axes.labelsize': 18,
    'legend.fontsize': 10,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.labelweight': 'normal',
})



# 可视化 Pick→Place 路径
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 拆分点对
pick_array = np.array([p for p, _ in valid_pairs])
place_array = np.array([q for _, q in valid_pairs])

# 可视化点
ax.scatter(pick_array[:, 0], pick_array[:, 1], pick_array[:, 2], c='green', label='Pick')
ax.scatter(place_array[:, 0], place_array[:, 1], place_array[:, 2], c='red', label='Place')

# 连接线（灰色）
lines = [[p, q] for p, q in valid_pairs]
line_collection = Line3DCollection(lines, colors='gray', linewidths=0.5, alpha=0.5)
ax.add_collection3d(line_collection)

# 加框函数（只画边框）
def draw_box(ax, xlim, ylim, zlim, color, label=None):
    x0, x1 = xlim
    y0, y1 = ylim
    z0, z1 = zlim
    drawn_label = False
    for z in [z0, z1]:
        ax.plot([x0, x1], [y0, y0], [z, z], color=color, label=label if not drawn_label else "")
        drawn_label = True
        ax.plot([x0, x1], [y1, y1], [z, z], color=color)
        ax.plot([x0, x0], [y0, y1], [z, z], color=color)
        ax.plot([x1, x1], [y0, y1], [z, z], color=color)
    for x in [x0, x1]:
        for y in [y0, y1]:
            ax.plot([x, x], [y, y], [z0, z1], color=color)

# 加框
draw_box(ax, [-25, -10], [10, 25], [6, 18], 'green', 'Pick Region')
draw_box(ax, [10, 25], [10, 25], [6, 18], 'red', 'Place Region')

# 设置轴
ax.set_xlabel('X (cm)')
ax.set_ylabel('Y (cm)')
ax.set_zlabel('Z (cm)')
# ax.set_title(f'Valid Pick→Place Paths with Lines (n={len(valid_pairs)} pairs)')
ax.view_init(elev=25, azim=135)
ax.xaxis.labelpad=8
ax.yaxis.labelpad=8
ax.zaxis.labelpad=8
ax.set_box_aspect(aspect=None, zoom=0.8)
# ax.legend()
plt.tight_layout()
plt.show()

from collections import Counter

print("🔍 Pick 点是否重复：", len(pick_points), "→ 去重后：", len(set(pick_points)))
print("🔍 Place 点是否重复：", len(place_points), "→ 去重后：", len(set(place_points)))

# 看看出现最多的是哪个
print("📌 Top Pick 重复：", Counter(pick_points).most_common(3))
print("📌 Top Place 重复：", Counter(place_points).most_common(3))

# 降维

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin_min
from mpl_toolkits.mplot3d.art3d import Line3DCollection

plt.rcParams.update({
    'font.size': 12,
    'font.family': 'Arial',
    'axes.titlesize': 14,
    'axes.labelsize': 18,
    'legend.fontsize': 10,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.labelweight': 'normal',
})

# — 原始路径对 ——
# valid_pairs 已在上文由 IK 验证得到
pair_vectors = np.array([np.hstack((p, q)) for p, q in valid_pairs])
print(f"✅ 6D 向量数量：{len(pair_vectors)}")

# — 1. 聚类中心计算 ——
n_clusters = min(1000, len(pair_vectors))
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, batch_size=1024)
labels = kmeans.fit_predict(pair_vectors)
centers = kmeans.cluster_centers_
print(f"✅ 聚类中心数量：{len(centers)}")

# — 2. 找到每个中心最近的原始路径索引 ——
closest_idx, _ = pairwise_distances_argmin_min(centers, pair_vectors)
print(f"✅ 代表性路径数量：{len(closest_idx)}")

# — 3. 提取代表性原始路径 ——
rep_pairs = [valid_pairs[i] for i in closest_idx]
pick_rep = np.array([p for p, _ in rep_pairs])
place_rep = np.array([q for _, q in rep_pairs])

# — 可视化 ——
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
# 原始簇中心替换为最近样本
ax.scatter(pick_rep[:, 0], pick_rep[:, 1], pick_rep[:, 2], c='green', label='Pick Rep')
ax.scatter(place_rep[:, 0], place_rep[:, 1], place_rep[:, 2], c='red', label='Place Rep')

# 连接线
lines = [[p, q] for p, q in zip(pick_rep, place_rep)]
line_collection = Line3DCollection(lines, colors='gray', linewidths=0.8, alpha=0.4)
ax.add_collection3d(line_collection)

# 工作空间框
def draw_box(ax, xlim, ylim, zlim, color):
    x0, x1 = xlim; y0, y1 = ylim; z0, z1 = zlim
    for z in (z0, z1):
        ax.plot([x0, x1], [y0, y0], [z, z], color=color)
        ax.plot([x0, x1], [y1, y1], [z, z], color=color)
        ax.plot([x0, x0], [y0, y1], [z, z], color=color)
        ax.plot([x1, x1], [y0, y1], [z, z], color=color)
    for x in (x0, x1):
        for y in (y0, y1):
            ax.plot([x, x], [y, y], [z0, z1], color=color)

draw_box(ax, PickR_Px, PickR_Py, PickR_Pz, 'green')
draw_box(ax, PlaceR_Px, PlaceR_Py, PlaceR_Pz, 'red')

ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)'); ax.set_zlabel('Z (cm)')
ax.view_init(elev=25, azim=135)
ax.xaxis.labelpad = 8; ax.yaxis.labelpad = 8; ax.zaxis.labelpad = 8
ax.set_box_aspect(aspect=None, zoom=0.8)
ax.legend()
plt.tight_layout()
plt.show()


## PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from scipy.spatial.distance import euclidean

import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'Arial',  # 或 'Times New Roman'
    'axes.titlesize': 14,
    'axes.labelsize': 18,
    'legend.fontsize': 14,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.labelweight': 'normal',
})

# ✅ Step 1: Prepare valid task pairs (6D vectors, rounded)
X = np.array([np.round(list(pick) + list(place), 2) for pick, place in valid_pairs])

# ✅ Step 2: Perform clustering to extract representative samples
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed).fit(X)
labels = kmeans.labels_
centers = np.round(kmeans.cluster_centers_, 2)  # ✅ 保留两位小数
print(centers.shape)

# ✅ Step 3: PCA 2D projection
pca = PCA(n_components=2)
X_2d = np.round(pca.fit_transform(X), 2)          # ✅ 保留两位小数
centers_2d = np.round(pca.transform(centers), 2)  # ✅ 保留两位小数

# -------------------------------------------
# ✅ Visualization 1: Colored clustering results
# -------------------------------------------
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab20', s=10, alpha=0.6, label='Clustered Points')
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c='black', s=25, marker='x', label='Cluster Centers')

# plt.title("PCA Projection with Clustering Labels")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend(loc='upper right')
plt.grid(True)
plt.colorbar(scatter, label="Cluster ID")
plt.show()

# -------------------------------------------
# ✅ Visualization 2: Pick→Place Distance Distribution
# -------------------------------------------
all_distances = [round(euclidean(p, q), 2) for p, q in valid_pairs]                   # ✅ 保留两位小数
cluster_distances = [round(euclidean(center[:3], center[3:]), 2) for center in centers]  # ✅ 保留两位小数

plt.figure(figsize=(10, 5))
plt.hist(all_distances, bins=40, alpha=0.5, label='Original Valid Pairs')
plt.hist(cluster_distances, bins=30, alpha=0.7, label='Clustered Representatives')
# plt.title("Pick→Place Distance Distribution (Before vs After Clustering)")
plt.xlabel("Pick→Place Euclidean Distance")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin_min
from scipy.spatial.distance import euclidean

plt.rcParams.update({
    'font.size': 12,
    'font.family': 'Arial',
    'axes.titlesize': 14,
    'axes.labelsize': 18,
    'legend.fontsize': 14,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.labelweight': 'normal',
})

# ✅ Step 1: Prepare valid task pairs (6D vectors, rounded)
X = np.array([np.round(list(pick) + list(place), 2) for pick, place in valid_pairs])

# ✅ Step 2: Perform clustering to extract representative samples
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed).fit(X)
labels = kmeans.labels_
centers = kmeans.cluster_centers_

# ✅ Step 2.1: Find closest original samples to each cluster center
closest_idx, _ = pairwise_distances_argmin_min(centers, X)
rep_pairs = [valid_pairs[i] for i in closest_idx]
rep_vectors = X[closest_idx]

# ✅ Step 3: PCA 2D projection
pca = PCA(n_components=2)
X_2d = np.round(pca.fit_transform(X), 2)
rep_2d = np.round(pca.transform(rep_vectors), 2)

# -------------------------------------------
# ✅ Visualization 1: Colored clustering results
# -------------------------------------------
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1],
                      c=labels, cmap='tab20', s=10, alpha=0.6, label='Clustered Points')
plt.scatter(rep_2d[:, 0], rep_2d[:, 1],
            c='black', s=25, marker='x', label='Selected Representatives')

plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend(loc='upper right')
plt.grid(True)
plt.colorbar(scatter, label="Cluster ID")
plt.show()

# -------------------------------------------
# ✅ Visualization 2: Pick→Place Distance Distribution
# -------------------------------------------
# Original distances
all_distances = [round(euclidean(p, q), 2) for p, q in valid_pairs]
# Distances for selected representatives
cluster_distances = [round(euclidean(p, q), 2) for p, q in rep_pairs]

plt.figure(figsize=(10, 5))
plt.hist(all_distances, bins=40, alpha=0.5, label='Original Valid Pairs')
plt.hist(cluster_distances, bins=30, alpha=0.7, label='Selected Representatives')
plt.xlabel("Pick→Place Euclidean Distance")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
from scipy.spatial.distance import euclidean
from sklearn.utils import shuffle  # ✅ 新增

# ✅ 解析聚类中心为 pick 和 place 点，并计算距离
samples = []
for c in centers:
    pick = tuple(c[:3])
    place = tuple(c[3:])
    distance = round(euclidean(pick, place), 2)
    samples.append((pick, place, distance))

# ✅ 保存为 DataFrame
df = pd.DataFrame([
    list(pick) + list(place) + [distance]
    for pick, place, distance in samples
], columns=['pick_x', 'pick_y', 'pick_z', 'place_x', 'place_y', 'place_z', 'distance'])

# ✅ 打乱行顺序
df = shuffle(df, random_state=seed)
print(df.head())

# ✅ 输出路径（你可以改名避免覆盖）
df.to_csv(f'output/representative_tasks_shuffled_{seed}_{n_clusters}.csv', index=False)
print("✅ Saved shuffled representative_tasks.csv")